In [1]:
%%capture

!pip install openai
!pip install alibabacloud_green20220302==2.2.11
!pip install aliyun-python-sdk-core-v3==2.13.10
!pip install -v aliyun-python-sdk-green==3.6.6
!pip install oss2
!pip install levenshtein


In [2]:
import pandas as pd
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from collections import Counter
import os
import random
from openai import OpenAI
from google.colab import userdata
import json
from alibabacloud_green20220302.client import Client
from alibabacloud_green20220302 import models
from alibabacloud_tea_openapi.models import Config
from google.colab import userdata # Import userdata to access Colab secrets
import time
import csv
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
def load_data(top_k=200):
    df_texts = pd.read_csv('/content/drive/MyDrive/thesis/processed_finished_data/Week18_masterdataset.csv')
    texts = df_texts.iloc[:, 0].dropna().astype(str).tolist()

    df_keywords = pd.read_csv('/content/drive/MyDrive/thesis/all_keywords.csv', on_bad_lines='skip')
    raw_keywords = df_keywords.iloc[:, 16].dropna().astype(str).tolist()

    keyword_counts = Counter(raw_keywords)
    most_common_keywords = [kw for kw, count in keyword_counts.most_common(top_k)]

    return texts, most_common_keywords

def get_context_representation(texts, keywords, model):  # model passed in, not created here
    v_sbert = model.encode(texts, batch_size=256, show_progress_bar=True)

    l_text = np.array([[len(t)] for t in texts])

    keywords_lower = [k.lower() for k in keywords]
    v_keywords = []
    for text in texts:
        text_lower = text.lower()
        presence = [1 if kw in text_lower else 0 for kw in keywords_lower]
        v_keywords.append(presence)

    v_keywords = np.array(v_keywords)
    x = np.concatenate([v_sbert, l_text, v_keywords], axis=1)
    return x

In [6]:
# Set your key here
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

class EvasionActionSpace:
    def __init__(self, api_key=None):
        print("Initializing Evasion Action Space (Powered by GPT-4o mini)...")
        # Use provided api_key or default from environment variable
        self.client = OpenAI(api_key=api_key or os.environ.get("OPENAI_API_KEY"))
        self.model = "gpt-4o-mini"

        #Rule-based assets for precise character attacks
        self.leet_map = {'a': '@', 'e': '3', 'i': '1', 'o': '0', 's': '$', 't': '7'}
        self.distractions = [
            "Today is a sunny day.", "The weather is nice.",
            "What is your favorite color?", "I am a student at Princeton University",
            "I like pizza."
        ]

    def execute_action(self, arm_index, text, temperature=0.7):
        """
        Routes the action to either GPT-4o mini (Semantic) or Python (Character).
        Use temperature=0.0 for deterministic outputs (e.g. during validation).
        """
        try:
            #TIER 1: SEMANTIC ACTIONS (GPT-4o mini) ---
            #Smart rewriting tasks

            if arm_index == 0: return self._llm_rewrite(text, "Replace words with synonyms. Keep meaning unchanged.", temperature)
            elif arm_index == 1: return self._llm_rewrite(text, "Replace specific nouns with general categories (e.g. 'Ferrari' -> 'car'). Keep meaning unchanged", temperature)
            elif arm_index == 2: return self._llm_rewrite(text, "Replace general nouns with specific examples (e.g. 'car' -> 'sedan'). Keep meaning unchanged", temperature)
            elif arm_index == 3: return self._llm_rewrite(text, "Reorder words/clauses. Keep meaning unchanged", temperature)
            elif arm_index == 4: return self._llm_rewrite(text, "Paraphrase this sentence completely. Keep meaning unchanged", temperature)
            elif arm_index == 5: return self._llm_rewrite(text, "Replace proper nouns/organizations with acronyms. Keep meaning unchanged", temperature)
            elif arm_index == 6: return self._llm_rewrite(text, "Insert neutral filler words (e.g. 'actually', 'simply'). Keep meaning unchanged", temperature)
            elif arm_index == 7: return self._llm_rewrite(text, "Switch voice: Active -> Passive, or Passive -> Active. Keep meaning unchanged", temperature)
            elif arm_index == 8: return self._llm_rewrite(text, "Use Tiktok speak and slang to rephrase the sentence if possible. Keep meaning unchanged", temperature)
            elif arm_index == 9: return self._llm_rewrite(text, "Rewrite text in a detached, academic, clinical tone. Use formal scholarly language. Keep meaning unchanged", temperature)
            elif arm_index == 10: return self._llm_rewrite(text, "Break text up into multiple shorter, individually innocuous sentence fragments. Split it so that no single fragment contains the full sensitive meaning on its own. Keep meaning unchanged.", temperature)
            elif arm_index == 11: return self._llm_rewrite(text, "Rewrite a few words to how they sound rather than how they are spelled (e.g. 'fruit', 'froot). Keep meaning unchanged", temperature)
            elif arm_index == 12: return self._llm_rewrite(text, "If possible, replace sensitive topics with code words or metaphors (e.g., 'baking bread', 'chemical process'). If impossible, do not change. Keep meaning unchanged", temperature)
            elif arm_index == 13: return self._llm_rewrite(text, "State that the opposite of what occured was false (e.g., 'violent protest,' 'protest was not unaggressive). Keep meaning unchanged", temperature)
            elif arm_index == 14: return self._llm_rewrite(text, "Rephrase the sentence using leetspeek. Keep meaning unchanged", temperature)
            #TIER 2: CHARACTER ACTIONS (Python Rules) ---
            #Precision tasks (LLMs are bad at these!)
            elif arm_index == 15: return self._distraction_augmentation(text)
            elif arm_index == 16: return self._homoglyph_substitution(text)

            else: return text

        except Exception as e:
            print(f"Action {arm_index} failed: {e}")
            return text

    def _llm_rewrite(self, text, instruction, temperature=0.7):
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a precise text rewriting engine. Output ONLY the rewritten text. No conversational filler."},
                    {"role": "user", "content": f"Instruction: {instruction}\nOriginal: \"{text}\""}
                ],
                temperature=temperature,
                max_tokens=200
            )
            return response.choices[0].message.content.strip().replace('"', '')
        except Exception as e:
            print(f"OpenAI Error: {e}")
            return text

    def _character_obfuscation(self, text):
        chars = list(text)
        candidates = [i for i, c in enumerate(chars) if c.lower() in self.leet_map]
        if not candidates: return text
        targets = random.sample(candidates, k=min(2, len(candidates)))
        for i in targets: chars[i] = self.leet_map[chars[i].lower()]
        return "".join(chars)

    def _distraction_augmentation(self, text):
        return text + " " + random.choice(self.distractions)

    def _homoglyph_substitution(self, text):
        map_ = {'a': 'а', 'e': 'е', 'o': 'о', 'p': 'р', 'x': 'х'}
        chars = list(text)
        for i, c in enumerate(chars):
            if c in map_ and random.random() < 0.3: chars[i] = map_[c]
        return "".join(chars)

    def _zero_width_injection(self, text):
        if " " not in text: return text
        words = text.split()
        idx = random.randint(0, len(words)-1)
        words[idx] = '\u200b'.join(list(words[idx]))
        return " ".join(words)

In [7]:
#1. ORACLE SETUP (Based on your notebook) ---
def setup_oracle_client():
    """
    Initializes the Alibaba Cloud Content Moderation Client.
    Uses credentials from your provided notebook.
    """
    config = Config(
        access_key_id=userdata.get('ALIBABA_KEY_ID'), # Use Colab secret
        access_key_secret=userdata.get('ALIBABA_KEY_SECRET'), # Use Colab secret
        connect_timeout=10000,
        read_timeout=3000,
        region_id='ap-southeast-1',
        endpoint='green-cip.ap-southeast-1.aliyuncs.com'
    )
    return Client(config)

def query_oracle(text, client):
    """
    Queries Alibaba Content Moderation and returns a tuple:
    (success_status, violation_label)

    success_status: 1 (Safe), 0 (Blocked)
    violation_label: The specific category (e.g., 'politics', 'porn', 'none')
    """
    serviceParameters = {
        'content': text
    }

    textModerationPlusRequest = models.TextModerationPlusRequest(
        service='comment_multilingual_pro_global',
        service_parameters=json.dumps(serviceParameters)
    )

    try:
        response = client.text_moderation_plus(textModerationPlusRequest)

        if response.status_code == 200 and hasattr(response, 'body') and response.body is not None:
            data = response.body.data

            # Default values
            success_status = 0
            violation_label = "unknown"

            if data is not None:
                risk_level = getattr(data, 'risk_level', None)

                # Check for success
                if risk_level == 'none':
                    success_status = 1
                    violation_label = "none"
                else:
                    success_status = 0
                    # Attempt to extract the first specific violation label
                    if hasattr(data, 'result') and data.result:
                        # Standard structure for TextModerationPlus result list
                        violation_label = data.result[0].label
                    elif hasattr(data, 'labels') and data.labels:
                        violation_label = data.labels[0].label
                    else:
                        violation_label = "unspecified_risk"

            return success_status, violation_label

        return 0, "api_error"

    except Exception as err:
        print(f"API Connection Exception: {err}")
        return 0, "connection_exception"

# --- REWARD FUNCTION (CHANGE #2) ---
# Failure = 0 penalty. This makes expected reward decompose as:
#   E[r | z, a] = P(evasion | z, a) * E[quality | z, a, evasion]
# Arms that never evade converge to zero. Arms that evade but mangle text
# get low positive values. Arms that evade cleanly get high positive values.
def calculate_reward(api_success, sim_score, edit_cost, lmbda_param=0.5):
    if api_success == 1:
        return sim_score - lmbda_param * edit_cost
    else:
        return 0.0


import numpy as np
import pickle
from sentence_transformers import SentenceTransformer, util
from Levenshtein import distance as lev_distance

#HELPER: METRIC CALCULATIONS ---
def get_metrics(original_text, original_embedding, modified_text, sbert_model):
    """
    Calculates reward components.
    Accepts a precomputed embedding for the original text instead of re-encoding it.
    """
    # 1. Semantic Similarity — only encode the modified text
    emb2 = sbert_model.encode(modified_text, convert_to_tensor=True)
    sim_score = util.pytorch_cos_sim(original_embedding, emb2).item()

    # 2. Normalized Edit Cost
    dist = lev_distance(original_text, modified_text)
    max_len = max(len(original_text), len(modified_text))
    norm_cost = 0.0 if max_len == 0 else dist / max_len

    return sim_score, norm_cost

In [8]:
# --- CHANGE #4: FROZEN BACKBONE + PURE LinUCB ---
# The backbone is pre-trained once (as a binary classifier: censored vs uncensored)
# and then FROZEN. LinUCB runs on the fixed latent features.
# This avoids the critical bug where backbone retraining invalidates A_inv/b/w matrices.

class Backbone(nn.Module):
    def __init__(self, input_dim=585, latent_dim=64):
        super(Backbone, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.network(x)


def pretrain_backbone(backbone, feature_vectors, active_training_indices, n_total, epochs=10, lr=1e-3, batch_size=128):
    """
    Pre-train backbone as a binary classifier: censored (1) vs uncensored (0).
    This gives the latent space meaningful structure before we freeze it.
    """
    classifier_head = nn.Linear(backbone.network[-2].out_features, 1)
    all_params = list(backbone.parameters()) + list(classifier_head.parameters())
    optimizer = optim.Adam(all_params, lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    # Build training data: positive = censored texts, negative = random uncensored texts
    censored_set = set(active_training_indices)
    positive_indices = list(censored_set)

    # Sample an equal number of uncensored indices
    all_indices = set(range(n_total))
    uncensored_indices = list(all_indices - censored_set)
    n_pos = len(positive_indices)
    negative_indices = random.sample(uncensored_indices, min(n_pos, len(uncensored_indices)))

    train_indices = positive_indices + negative_indices
    train_labels = [1.0] * len(positive_indices) + [0.0] * len(negative_indices)

    # Shuffle
    combined = list(zip(train_indices, train_labels))
    random.shuffle(combined)
    train_indices, train_labels = zip(*combined)
    train_indices = list(train_indices)
    train_labels = list(train_labels)

    backbone.train()
    classifier_head.train()

    for epoch in range(epochs):
        perm = list(range(len(train_indices)))
        random.shuffle(perm)
        epoch_loss = 0.0
        n_batches = 0

        for start in range(0, len(perm), batch_size):
            batch_perm = perm[start:start + batch_size]
            batch_idx = [train_indices[p] for p in batch_perm]
            batch_labels = [train_labels[p] for p in batch_perm]

            x = torch.FloatTensor(feature_vectors[batch_idx])
            y = torch.FloatTensor(batch_labels).view(-1, 1)

            optimizer.zero_grad()
            z = backbone(x)
            logits = classifier_head(z)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        print(f"  Pretrain epoch {epoch+1}/{epochs}, loss={epoch_loss/n_batches:.4f}")

    # Freeze the backbone after pre-training
    backbone.eval()
    for param in backbone.parameters():
        param.requires_grad = False

    print("Backbone pre-trained and FROZEN.")
    return backbone


class FrozenLinUCBBandit:
    """
    LinUCB with a frozen pre-trained backbone.
    No backbone retraining — the latent space is fixed, so A_inv/b/w
    matrices remain valid throughout training.
    """
    def __init__(self, backbone, n_arms=18, m=64, alpha=1.0, lmbda=1.0):
        self.n_arms = n_arms
        self.latent_dim = m
        self.alpha = alpha
        self.backbone = backbone  # already frozen

        self.w = [torch.zeros(m, 1) for _ in range(n_arms)]
        self.A_inv = [torch.eye(m) / lmbda for _ in range(n_arms)]
        self.b = [torch.zeros(m, 1) for _ in range(n_arms)]

    def get_latent(self, x):
        with torch.no_grad():
            z = self.backbone(torch.FloatTensor(x))
        z = z / (z.norm() + 1e-8)  # normalize to unit norm
        return z.reshape(-1, 1)

    def select_arm(self, x, alpha_override=None):
        """Select arm. Use alpha_override=0 for pure exploitation (validation)."""
        z = self.get_latent(x)
        alpha = alpha_override if alpha_override is not None else self.alpha
        p = np.zeros(self.n_arms)

        for a in range(self.n_arms):
            mu = torch.mm(self.w[a].t(), z)
            # Clamp to prevent numerical issues from Sherman-Morrison drift
            sigma = torch.sqrt(torch.clamp(torch.mm(torch.mm(z.t(), self.A_inv[a]), z), min=1e-8))
            p[a] = mu + alpha * sigma

        return np.argmax(p), z

    def update(self, arm_idx, x, reward, z=None):
        if z is None:
            z = self.get_latent(x)

        v = torch.mm(self.A_inv[arm_idx], z)
        self.A_inv[arm_idx] -= torch.mm(v, v.t()) / (1 + torch.mm(z.t(), v))
        self.b[arm_idx] += reward * z
        self.w[arm_idx] = torch.mm(self.A_inv[arm_idx], self.b[arm_idx])

In [18]:
CANDIDATE_SIZE = 20000
NUM_EPOCHS = 5           # CHANGE #5: multiple passes over training data
VAL_EVERY = 100          # CHANGE #1: validate every N training episodes
VAL_SET_SIZE = 100       # CHANGE #1: number of frozen validation texts

oracle_client = setup_oracle_client()
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
api_key = userdata.get("OPENAI_API_KEY")
evasion_space = EvasionActionSpace(api_key=api_key)

history_path = '/content/drive/MyDrive/thesis/pkls/running/new_run_apr/training_history_log.csv'
model_path = '/content/drive/MyDrive/thesis/pkls/running/new_run_apr/trained_frozenlinucb_model.pkl'
prefilter_path = '/content/drive/MyDrive/thesis/pkls/running/new_run_apr/prefiltered_training_set.pkl'
context_path = '/content/drive/MyDrive/thesis/pkls/running/new_run_apr/context_features.pkl'
val_log_path = '/content/drive/MyDrive/thesis/pkls/running/new_run_apr/validation_curve.csv'

raw_texts, top_k_keywords = load_data(top_k=200)

print("Loading feature vectors...")
with open(context_path, 'rb') as f:
    feature_vectors = pickle.load(f)

print(f"Loaded {len(raw_texts)} texts. Starting Pre-filtering...")

if os.path.exists(prefilter_path):
    with open(prefilter_path, 'rb') as f:
        cached = pickle.load(f)

    if len(cached) == 4:
        cached_size, active_training_indices, active_training_texts, initial_labels = cached
    else:
        cached_size = -1
        active_training_indices, active_training_texts, initial_labels = [], [], []

    if CANDIDATE_SIZE > cached_size and cached_size != -1:
        print(f"Candidate size increased ({cached_size} -> {CANDIDATE_SIZE}). Processing new items...")

        new_candidate_texts = raw_texts[cached_size:CANDIDATE_SIZE]

        for i, text in tqdm(enumerate(new_candidate_texts, start=cached_size), total=len(new_candidate_texts), desc="Pre-filtering"):
            is_safe, label = query_oracle(text, oracle_client)
            if is_safe == 0:
                active_training_indices.append(i)
                active_training_texts.append(text)
                initial_labels.append(label)

        with open(prefilter_path, 'wb') as f:
            pickle.dump((CANDIDATE_SIZE, active_training_indices, active_training_texts, initial_labels), f)
        print(f"Pre-filtering complete. {len(active_training_indices)} total blocked texts cached.")

    elif CANDIDATE_SIZE < cached_size and cached_size != -1:
        print(f"Candidate size decreased ({cached_size} -> {CANDIDATE_SIZE}). Truncating cache...")
        valid_items = [(idx, txt, lbl) for idx, txt, lbl in zip(active_training_indices, active_training_texts, initial_labels) if idx < CANDIDATE_SIZE]

        if valid_items:
            active_training_indices, active_training_texts, initial_labels = map(list, zip(*valid_items))
        else:
            active_training_indices, active_training_texts, initial_labels = [], [], []

        with open(prefilter_path, 'wb') as f:
            pickle.dump((CANDIDATE_SIZE, active_training_indices, active_training_texts, initial_labels), f)
        print(f"Cache truncated. {len(active_training_indices)} blocked texts remaining.")

    else:
        print(f"Loaded {len(active_training_indices)} pre-filtered texts from cache.")

else:
    print("No valid cache found. Running pre-filter from scratch...")
    active_training_indices, active_training_texts, initial_labels = [], [], []

    candidate_texts = raw_texts[:CANDIDATE_SIZE]
    for i, text in tqdm(enumerate(candidate_texts), total=len(candidate_texts), desc="Pre-filtering"):
        is_safe, label = query_oracle(text, oracle_client)
        if is_safe == 0:
            active_training_indices.append(i)
            active_training_texts.append(text)
            initial_labels.append(label)

    with open(prefilter_path, 'wb') as f:
        pickle.dump((CANDIDATE_SIZE, active_training_indices, active_training_texts, initial_labels), f)
    print(f"Pre-filtering complete. {len(active_training_indices)} blocked texts cached.")

# --- CHANGE #1: FREEZE A FIXED VALIDATION SET ---
all_tasks = list(zip(active_training_indices, active_training_texts, initial_labels))
random.seed(42)
random.shuffle(all_tasks)

val_tasks = all_tasks[:VAL_SET_SIZE]
train_tasks = all_tasks[VAL_SET_SIZE:]

print(f"Validation set: {len(val_tasks)} texts (frozen)")
print(f"Training set: {len(train_tasks)} texts")

# --- CHANGE #4: PRE-TRAIN AND FREEZE BACKBONE ---
backbone = Backbone(input_dim=585, latent_dim=64)

if os.path.exists(model_path):
    print(f"Loading existing model from {model_path}...")
    try:
        with open(model_path, 'rb') as f:
            bandit = pickle.load(f)
    except Exception as e:
        print(f"Failed to load model: {e}")
        print("Pre-training backbone and initializing fresh bandit...")
        backbone = pretrain_backbone(backbone, feature_vectors, active_training_indices, n_total=len(feature_vectors))
        bandit = FrozenLinUCBBandit(backbone, n_arms=18, m=64, alpha=1, lmbda=1.0)
else:
    print("No existing model found. Pre-training backbone...")
    backbone = pretrain_backbone(backbone, feature_vectors, active_training_indices, n_total=len(feature_vectors))
    bandit = FrozenLinUCBBandit(backbone, n_arms=18, m=64, alpha=1, lmbda=1.0)

# Precompute embeddings for ALL texts (train + val) for reward computation
print("Precomputing original text embeddings...")
all_task_indices = [idx for idx, _, _ in all_tasks]
all_task_texts = [text for _, text, _ in all_tasks]
embeddings_list = sbert_model.encode(all_task_texts, batch_size=256,
                                    convert_to_tensor=True, show_progress_bar=True)
original_embeddings = dict(zip(all_task_indices, embeddings_list))


# --- VALIDATION FUNCTION ---
def run_validation(bandit, val_tasks, feature_vectors, original_embeddings,
                   evasion_space, oracle_client, sbert_model):
    rng_state = random.getstate()
    random.seed(999)

    val_successes = 0
    val_rewards = []

    for idx, text, label in val_tasks:
        context_vector = feature_vectors[idx]
        action_idx, z = bandit.select_arm(context_vector, alpha_override=0.0)
        modified_text = evasion_space.execute_action(action_idx, text, temperature=0.0)

        api_success, _ = query_oracle(modified_text, oracle_client)
        orig_emb = original_embeddings[idx]
        sim_score, edit_cost = get_metrics(text, orig_emb, modified_text, sbert_model)
        reward = calculate_reward(api_success, sim_score, edit_cost)

        val_successes += api_success
        val_rewards.append(reward)

    random.setstate(rng_state)

    evasion_rate = val_successes / len(val_tasks)
    avg_reward = np.mean(val_rewards)
    return evasion_rate, avg_reward


# --- DETERMINISTIC PER-EPOCH SHUFFLE ---
def get_epoch_order(train_tasks, epoch):
    """Deterministic shuffle per epoch so order is reproducible across restarts."""
    tasks_copy = list(train_tasks)
    random.seed(1000 + epoch)
    random.shuffle(tasks_copy)
    return tasks_copy


# --- RECONSTRUCT RESUME STATE FROM CSV ---
def reconstruct_from_csv(history_path):
    """
    Parse training_history_log.csv to figure out where training left off.
    Returns (global_step, processed_per_epoch, total_rewards)
    where processed_per_epoch is {epoch_0indexed: set of original_index}.
    """
    if not os.path.isfile(history_path):
        return 0, {}, []

    import pandas as pd
    df = pd.read_csv(history_path)

    if len(df) == 0:
        return 0, {}, []

    global_step = int(df['episode'].max())
    total_rewards = df['reward'].tolist()

    # CSV epoch is 1-indexed → convert to 0-indexed
    processed_per_epoch = {}
    for epoch_1idx, group in df.groupby('epoch'):
        epoch_0idx = int(epoch_1idx) - 1
        processed_per_epoch[epoch_0idx] = set(group['original_index'].astype(int).tolist())

    return global_step, processed_per_epoch, total_rewards


# --- TRAINING LOOP WITH CSV-BASED RESUMPTION ---
print("Starting Online Training ---")

LAMBDA_PARAM = 0.5
training_failed = False

global_step, processed_per_epoch, total_rewards = reconstruct_from_csv(history_path)

if global_step > 0:
    print(f"Reconstructed state from CSV: global_step={global_step}")
    for ep, indices in sorted(processed_per_epoch.items()):
        print(f"  Epoch {ep+1}: {len(indices)}/{len(train_tasks)} items completed")
else:
    print("No prior training history found. Starting fresh.")

# Initialize validation log
val_file_exists = os.path.isfile(val_log_path)
val_log_file = open(val_log_path, mode='a', newline='', encoding='utf-8')
val_writer = csv.DictWriter(val_log_file, fieldnames=['episode', 'evasion_rate', 'avg_reward'])
if not val_file_exists:
    val_writer.writeheader()

file_exists = os.path.isfile(history_path)

with open(history_path, mode='a', newline='', encoding='utf-8') as csvfile:
    fieldnames = [
        'episode', 'epoch', 'timestamp', 'duration', 'original_index', 'action_idx',
        'initial_label', 'final_label', 'success', 'sim_score',
        'edit_cost', 'reward', 'original_text', 'modified_text'
    ]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    if not file_exists:
        writer.writeheader()

    for epoch in range(NUM_EPOCHS):
        if training_failed:
            break

        # Deterministic per-epoch shuffle (reproducible across restarts)
        epoch_tasks = get_epoch_order(train_tasks, epoch)

        # Figure out which items to skip based on CSV history
        already_done = processed_per_epoch.get(epoch, set())

        if len(already_done) >= len(epoch_tasks):
            print(f"\n=== EPOCH {epoch+1}/{NUM_EPOCHS} — already completed ({len(already_done)} items), skipping ===")
            continue

        # Filter out already-processed items for this epoch
        remaining_tasks = [(idx, txt, lbl) for idx, txt, lbl in epoch_tasks
                           if idx not in already_done]

        if len(already_done) > 0:
            print(f"\n=== EPOCH {epoch+1}/{NUM_EPOCHS} ({len(epoch_tasks)} texts, "
                  f"{len(already_done)} already done, {len(remaining_tasks)} remaining) ===")
        else:
            print(f"\n=== EPOCH {epoch+1}/{NUM_EPOCHS} ({len(epoch_tasks)} training texts) ===")

        for t, (original_idx, text, init_label) in enumerate(remaining_tasks):
            try:
                start_time = time.time()
                context_vector = feature_vectors[original_idx]

                action_idx, z = bandit.select_arm(context_vector)
                modified_text = evasion_space.execute_action(action_idx, text)

                api_success, violation_label = query_oracle(modified_text, oracle_client)
                status = "PASSED" if api_success == 1 else "BLOCKED"

                orig_emb = original_embeddings[original_idx]
                sim_score, edit_cost = get_metrics(text, orig_emb, modified_text, sbert_model)
                reward = calculate_reward(api_success, sim_score, edit_cost, lmbda_param=LAMBDA_PARAM)

                bandit.update(action_idx, context_vector, reward, z=z)
                total_rewards.append(reward)
                global_step += 1

                duration = time.time() - start_time

                log_entry = {
                    'episode': global_step,
                    'epoch': epoch + 1,
                    'timestamp': time.time(),
                    'duration': duration,
                    'original_index': original_idx,
                    'action_idx': action_idx,
                    'initial_label': init_label,
                    'final_label': violation_label,
                    'success': api_success,
                    'sim_score': sim_score,
                    'edit_cost': edit_cost,
                    'reward': reward,
                    'original_text': text,
                    'modified_text': modified_text
                }

                writer.writerow(log_entry)

                # Heartbeat every 10 steps
                if global_step % 10 == 0:
                    print(f"  [Step {global_step:>5} | Epoch {epoch+1}] Arm={action_idx:>2} | {status} | Sim={sim_score:.3f} | Cost={edit_cost:.3f} | R={reward:.3f} | {duration:.1f}s")

                # --- PERIODIC VALIDATION ---
                if global_step % VAL_EVERY == 0:
                    csvfile.flush()
                    os.fsync(csvfile.fileno())

                    print(f"\n--- Validation at step {global_step} ---")
                    evasion_rate, avg_val_reward = run_validation(
                        bandit, val_tasks, feature_vectors, original_embeddings,
                        evasion_space, oracle_client, sbert_model
                    )
                    print(f"  Evasion rate: {evasion_rate:.3f} | Avg reward: {avg_val_reward:.3f}")

                    val_writer.writerow({
                        'episode': global_step,
                        'evasion_rate': evasion_rate,
                        'avg_reward': avg_val_reward
                    })
                    val_log_file.flush()

                    with open(model_path, 'wb') as f:
                        pickle.dump(bandit, f)

                if (global_step) % 50 == 0:
                    print("-" * 60)
                    print(f"Epoch {epoch+1} | Step {global_step} (Index {original_idx})")
                    print(f"Action:   {action_idx}")
                    print(f"Original: {text[:80]}...")
                    print(f"Metrics:  {status} | Sim={sim_score:.3f} | Cost={edit_cost:.3f} | Reward={reward:.3f}")

            except Exception as e:
                print(f"Critical error at index {original_idx}: {e}")
                with open(model_path, 'wb') as f:
                    pickle.dump(bandit, f)
                training_failed = True
                break

val_log_file.close()

with open(model_path, 'wb') as f:
    pickle.dump(bandit, f)

print("\nSession Complete ---")
if total_rewards:
    print(f"Total episodes: {global_step}")
    print(f"Average Reward: {np.mean(total_rewards):.3f}")
else:
    print("No episodes ran this session.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initializing Evasion Action Space (Powered by GPT-4o mini)...


/tmp/ipykernel_7884/139008619.py:5: DtypeWarning: Columns (99) have mixed types. Specify dtype option on import or set low_memory=False.
  df_keywords = pd.read_csv('/content/drive/MyDrive/thesis/all_keywords.csv', on_bad_lines='skip')


Loading feature vectors...
Loaded 586592 texts. Starting Pre-filtering...
Loaded 1924 pre-filtered texts from cache.
Validation set: 100 texts (frozen)
Training set: 1824 texts
Loading existing model from /content/drive/MyDrive/thesis/pkls/running/new_run_apr/trained_frozenlinucb_model.pkl...
Precomputing original text embeddings...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Starting Online Training ---
Reconstructed state from CSV: global_step=9100
  Epoch 1: 1824/1824 items completed
  Epoch 2: 1824/1824 items completed
  Epoch 3: 1824/1824 items completed
  Epoch 4: 1824/1824 items completed
  Epoch 5: 1804/1824 items completed

=== EPOCH 1/5 — already completed (1824 items), skipping ===

=== EPOCH 2/5 — already completed (1824 items), skipping ===

=== EPOCH 3/5 — already completed (1824 items), skipping ===

=== EPOCH 4/5 — already completed (1824 items), skipping ===

=== EPOCH 5/5 (1824 texts, 1804 already done, 20 remaining) ===
  [Step  9110 | Epoch 5] Arm=12 | BLOCKED | Sim=0.586 | Cost=0.265 | R=0.000 | 2.1s
  [Step  9120 | Epoch 5] Arm= 8 | PASSED | Sim=0.747 | Cost=0.567 | R=0.463 | 3.6s

Session Complete ---
Total episodes: 9120
Average Reward: 0.292


In [ ]:
# @title
import pandas as pd
import numpy as np
from tqdm import tqdm

best_arms_results = []

print(f"Evaluating {len(val_tasks)} holdout texts across all {bandit.n_arms} arms...")

# Seed RNG for reproducible rule-based actions
rng_state = random.getstate()
random.seed(999)

for original_idx, text, init_label in tqdm(val_tasks):
    best_arm = -1
    best_reward = -1.0
    best_modified_text = ""
    best_sim = 0.0
    best_cost = 0.0
    best_success = 0

    for arm_idx in range(bandit.n_arms):
        # Execute action with temperature 0 for determinism
        modified_text = evasion_space.execute_action(arm_idx, text, temperature=0.0)

        # Query oracle
        api_success, violation_label = query_oracle(modified_text, oracle_client)

        # Get original embedding
        orig_emb = original_embeddings[original_idx]

        # Calculate metrics and reward
        sim_score, edit_cost = get_metrics(text, orig_emb, modified_text, sbert_model)
        reward = calculate_reward(api_success, sim_score, edit_cost, lmbda_param=LAMBDA_PARAM)

        # Update best arm
        if reward > best_reward:
            best_reward = reward
            best_arm = arm_idx
            best_modified_text = modified_text
            best_sim = sim_score
            best_cost = edit_cost
            best_success = api_success

    best_arms_results.append({
        'original_index': original_idx,
        'original_text': text,
        'initial_label': init_label,
        'best_arm': best_arm,
        'best_reward': best_reward,
        'success': best_success,
        'sim_score': best_sim,
        'edit_cost': best_cost,
        'best_modified_text': best_modified_text
    })

random.setstate(rng_state)

# Convert to DataFrame and save/display
df_best_arms = pd.DataFrame(best_arms_results)
output_path = '/content/drive/MyDrive/thesis/pkls/running/holdout_best_arms.csv'
df_best_arms.to_csv(output_path, index=False)
print(f"\nSaved best arms for holdout set to: {output_path}")
df_best_arms.head()